In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel

In [ ]:
first_data = pd.read_csv("/content/i2b2_2010_annotated_filtered.csv")
first_data = first_data.astype(str)
second_data = pd.read_csv("/content/uml_desc.csv")
second_data = second_data.astype(str)

In [ ]:
first_data = first_data[['Word','Tag']]
second_data = second_data[['Clinician_Friendly_Name','Entity_type']]

In [ ]:
print(first_data['Tag'].value_counts())

Tag
I-problem    23495
B-problem    16777
Name: count, dtype: int64


In [ ]:
joint_entities = []
single_entities = []
current_entity = []

for i in range(len(first_data)):
    if first_data['Tag'][i].startswith('B-'):
        if current_entity:
            joint_entities.append(" ".join(current_entity))
            current_entity = []
        current_entity.append(first_data['Word'][i])
    elif first_data['Tag'][i].startswith('I-'):
        current_entity.append(first_data['Word'][i])
    else:
        if current_entity:
            if len(current_entity) > 1:
                joint_entities.append(" ".join(current_entity))
            else:
                single_entities.append(current_entity[0])
            current_entity = []
if current_entity:
    if len(current_entity) > 1:
        joint_entities.append(" ".join(current_entity))
    else:
        single_entities.append(current_entity[0])


In [ ]:
# Ensure single entities are correctly extracted
for i in range(len(first_data) - 1):
    if first_data['Tag'][i].startswith('B-') and not first_data['Tag'][i + 1].startswith('I-'):
        single_entities.append(first_data['Word'][i])

# Remove single entities that are part of joint entities
joint_entity_words = set(word for entity in joint_entities for word in entity.split())
single_entities = [entity for entity in single_entities if entity not in joint_entity_words]

print("Joint Entities:", joint_entities)
print("Single Entities:", single_entities)

Joint Entities: ['bleeding vessel', 'dm2', 'cad', 'DVT/PE', 'ulcerative colitis', 'brbpr', 'lower abdominal pain', 'a symptom', 'a UTI', 'a large , bloody bowel movement', 'hypovolemic', 'hemoconcentrated', 'clot', 'a large , bloody bowel movement', 'CP', 'SOB', 'abd pain', 'continued BRBPR', 'DM-2', 'CAD', 'CHF', 'Right parietal intracranial bleeding', 'LBBB', 'Sinus node dysfunction', 'DVT', 'PE', 'Ulcerative colitis', 'NAD', 'anicteric sclera', 'jvd', 'm/r/g', 'rales', 'minimally distended', 'nt', 'rebound/guarding', 'trace edema at ankles', 'cyanosis', 'Increased tracer activity', 'active bleeding', 'colitis', 'other bowel pathology', "the patient 's bright red blood per rectum", 'left adrenal fat-containing lesion', 'a myelolipoma', 'hypodense lesion within the caudate lobe', 'Subcentimeter attenuation lesion within the lower pole', '3-mm noncalcified pulmonary nodule within the right lower lobe', 'a primary malignancy', 'dm2', 'cad', 'chf', 'uc', 'lower abdominal pain', 'brpbr', 

In [ ]:
print(len(joint_entities))
print(len(single_entities))

16776
1


In [ ]:
def divide_list(lst, n):
    # Determine the number of items per part
    part_size = len(lst) // n
    remainder = len(lst) % n

    # Create the parts
    parts = []
    start = 0
    for i in range(n):
        end = start + part_size + (1 if i < remainder else 0)
        part = lst[start:end]
        parts.append(part)
        start = end

    return parts

# Divide the list into 10 parts
ent = divide_list(joint_entities, 100)
for i, part in enumerate(ent):
    print(f"Part {i + 1}: {part}")

Part 1: ['bleeding vessel', 'dm2', 'cad', 'DVT/PE', 'ulcerative colitis', 'brbpr', 'lower abdominal pain', 'a symptom', 'a UTI', 'a large , bloody bowel movement', 'hypovolemic', 'hemoconcentrated', 'clot', 'a large , bloody bowel movement', 'CP', 'SOB', 'abd pain', 'continued BRBPR', 'DM-2', 'CAD', 'CHF', 'Right parietal intracranial bleeding', 'LBBB', 'Sinus node dysfunction', 'DVT', 'PE', 'Ulcerative colitis', 'NAD', 'anicteric sclera', 'jvd', 'm/r/g', 'rales', 'minimally distended', 'nt', 'rebound/guarding', 'trace edema at ankles', 'cyanosis', 'Increased tracer activity', 'active bleeding', 'colitis', 'other bowel pathology', "the patient 's bright red blood per rectum", 'left adrenal fat-containing lesion', 'a myelolipoma', 'hypodense lesion within the caudate lobe', 'Subcentimeter attenuation lesion within the lower pole', '3-mm noncalcified pulmonary nodule within the right lower lobe', 'a primary malignancy', 'dm2', 'cad', 'chf', 'uc', 'lower abdominal pain', 'brpbr', 'BRBPR',

In [ ]:
len(ent[0])

503

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=10)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load BioBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1")

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=10)
    with torch.no_grad():  # Disable gradient calculations
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load SciBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
model = AutoModel.from_pretrained("allenai/scibert_scivocab_uncased")

def get_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=10)
    with torch.no_grad():  # Disable gradient calculations
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/228k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

In [ ]:
# Generate embeddings for joint entities
joint_entity_embeddings = {entity: get_embedding(entity) for entity in joint_entities}

In [ ]:
print(len(joint_entity_embeddings))

3457


In [ ]:
single_entity_embeddings = {entity: get_embedding(entity) for entity in single_entities}

In [ ]:
# Generate embeddings for second file entities
second_entity_embeddings = {entity: get_embedding(entity) for entity in second_data['Clinician_Friendly_Name']}

KeyboardInterrupt: 

In [ ]:
def divide_dict(d, n):
    # Get list of dictionary items (key-value pairs)
    items = list(d.items())

    # Determine the number of items per part
    part_size = len(items) // n
    remainder = len(items) % n

    # Create the parts
    parts = []
    start = 0
    for i in range(n):
        end = start + part_size + (1 if i < remainder else 0)
        part = dict(items[start:end])
        parts.append(part)
        start = end

    return parts

# Divide the dictionary into 10 parts
parts = divide_dict(joint_entity_embeddings, 100)


In [ ]:
len(parts[0])

88

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def find_matches(entities, entity_embeddings, second_entity_embeddings, threshold=0.85):
    matches = {}
    for entity, embedding in entity_embeddings.items():
        max_similarity = 0
        best_match = None
        for second_entity, second_embedding in second_entity_embeddings.items():
            similarity = cosine_similarity(embedding, second_embedding)[0][0]
            if similarity > max_similarity:
                max_similarity = similarity
                best_match = second_entity
        matches[entity] = (max_similarity >= threshold, best_match, max_similarity)
    return matches

# Find matches for joint entities
joint_entity_matches = find_matches(ent[0], parts[0], second_entity_embeddings)

print("Joint Entity Matches:", joint_entity_matches)

In [ ]:
from google.colab import files
# Convert the dictionary into a list of tuples for DataFrame creation
data_tuples = [(key, *value) for key, value in joint_entity_matches.items()]

# Create a DataFrame with entity, match, matched_entity, score columns
df = pd.DataFrame(data_tuples, columns=['Entity', 'Match', 'Matched Entity', 'Score'])

df.to_csv('entity_linking_1.csv', index=False)
files.download('entity_linking_1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
def find_matches(entities, entity_embeddings, second_entity_embeddings, threshold=0.85):
    matches = {}
    for entity, embedding in entity_embeddings.items():
        max_similarity = 0
        best_match = None
        for second_entity, second_embedding in second_entity_embeddings.items():
            similarity = cosine_similarity(embedding, second_embedding)[0][0]
            if similarity > max_similarity:
                max_similarity = similarity
                best_match = second_entity
        matches[entity] = (max_similarity >= threshold, best_match, max_similarity)
    return matches

In [ ]:
from google.colab import files
lst = ['entity_linking_1.csv', 'entity_linking_2.csv', 'entity_linking_3.csv', 'entity_linking_4.csv',
       'entity_linking_5.csv', 'entity_linking_6.csv', 'entity_linking_7.csv', 'entity_linking_8.csv',
       'entity_linking_9.csv', 'entity_linking_10.csv']
j=0
for i in range (0,10):
# Find matches for joint entities
  joint_entity_matches = find_matches(ent[i], parts[i], second_entity_embeddings)
  print("Joint Entity Matches:", joint_entity_matches)
  data_tuples = [(key, *value) for key, value in joint_entity_matches.items()]
  df = pd.DataFrame(data_tuples, columns=['Entity', 'Match', 'Matched Entity', 'Score'])
  df.to_csv(lst[j], index=False)
  j+=1

In [ ]:
import pandas as pd
from google.colab import files

joint_entity_matches = {'biting tongue': (True, 'Tongue swelling', 0.9443403), 'cold / chills': (True, 'Chills', 0.92722625), 'the tongue injury': (True, 'Tongue laceration', 0.93974936), 'Urinary frequency': (True, 'Urinary frequency', 0.99999994), 'Mitral valve prolapse': (True, 'Mitral valve prolapse', 1.0000001), 'her rigidity': (True, 'Stiffness of joint', 0.8935854), 'groggy': (True, 'Black piedra', 0.9097166), 'laceration on the right edge': (True, 'Open wound of right ankle', 0.9550044), 'post laparoscopic incisions': (True, 'Incisional hernia', 0.92322075), 'inflammation in the right upper quadrant': (True, 'Localized swelling on right lower leg', 0.9236598), 'small fluid in the circumferential thickening of the distal stomach': (True, 'Mass of soft tissue of bilat legs', 0.89796823), 'a small ulcer': (True, 'Acute antral ulcer', 0.9190269), 'a bile leak': (True, 'Bile duct injury', 0.9363805), 'a bile leak from the duct of Luschka': (True, 'Bile duct rupture', 0.9323273), 'waxing and waning fevers': (True, 'Fever w chills', 0.86924624), 'really elevated': (True, 'Elevated CEA', 0.8863454), 'marked rebound': (True, 'Rebound headache', 0.927655), 'mild pancreatitis': (True, 'Acute pancreatitis', 0.9657953), 'intra-abdominal fluid collections': (True, 'Intraabdominal abscess', 0.93120015), "the patient 's pain": (True, 'Chronic pain due to trauma', 0.8831432), 'the rebound': (True, 'Rebound headache', 0.92629224), 'further rebound': (True, 'Rebound headache', 0.9400641), 'temperature spikes': (True, 'Heat stroke', 0.90444136), 'an anion gap acidosis': (True, 'Mixed acid base balance disorder', 0.9136729), 'ketoacidosis': (True, 'Ketoacidosis due to acute alcohol intoxication, unspecified type of alcohol use', 0.96546793), 'her anion gap acidosis': (True, 'Mixed acid base balance disorder w hypercapnia', 0.90348333), 'Post endoscopic retrograde cholangiopancreatography pancreatitis': (True, 'Hx of laparoscopic gastric banding device (removed)', 0.9172459), 'Anion gap acidosis': (True, 'Mixed acid base balance disorder', 0.9039383), 'Bile leak': (True, 'Bile duct injury', 0.94674844), "Berkitt's lymphoma": (True, 'Cutaneous T-cell lymphoma, unspecified type and site', 0.9254445), 'Liver lesions': (True, 'Liver disease', 0.9701688), 'the previously noted enumerable hepatic cyst in her liver': (True, 'Drug induced cholestatic hepatitis', 0.8956375), 'these cysts': (True, 'Bilat breast cysts', 0.9416186), 'hepatic candidiasis': (True, 'Retinal candidiasis', 0.95962703), "Burkitt's": (True, 'Burkitt lymphoma, extranodal and solid organs', 0.9215718), 'her hepatic candidiasis': (True, 'Penile candidiasis', 0.9462024), 'Guarded': (True, 'Canities', 0.8793113), "Burkitt's lymphoma": (True, 'Burkitt lymphoma, extranodal and solid organs', 0.9658275), 'Hepatic candidiasis': (True, 'Retinal candidiasis', 0.95962703), 'a cystic lesion': (True, 'Dermoid cyst, left ovary', 0.9277081), 'an enlargement of the head of his pancreas': (True, 'Acquired deformity of abdominal wall', 0.9048092), 'early pseudocyst': (True, 'Bilat ear pseudocyst', 0.9444542), 'significantly elevated alkaline phosphatase': (True, 'Elevated alkaline phosphatase', 0.9664953), 'his pancreatitis': (True, 'Acute pancreatitis', 0.966847), '1900-01-00 bandemia': (False, 'Hgb H constant spring disease', 0.8496969), 'no flow': (True, 'Persistent miosis', 0.87328076), 'common bile duct obstruction': (True, 'Bile duct obstruction', 0.9896699), 'recorded melena': (True, 'Melena', 0.9241893), 'a long nipple like papilla': (True, 'Bilat chronic giant papillary conjunctivitis', 0.9183203), 'Streptococcus sanguis-gordonii': (True, 'Streptococcal pharyngitis', 0.964356), 'Right rotator cuff injury': (True, 'Right rotator cuff injury', 1.0000002), 'Recent pancreatitis': (True, 'Chronic pancreatitis', 0.9748138), 'recurrent pancreatitis': (True, 'Acute pancreatitis', 0.9667306), 'Streptococcemia': (True, 'Streptococcal pharyngitis', 0.9699687), 'a psychiatric condition': (True, 'Psychological disorder', 0.90758437), 'Munos sign': (True, 'Leighs disease', 0.9191661), 'trace glucose': (True, 'Bandemia', 0.8988707), 'ketones': (True, 'Ketone metabolism disorder', 0.8639479), 'red cells': (True, 'Whole blood donor', 0.90706676), 'white cells': (True, 'Microcytosis', 0.9040838), 'mild biliary stricture': (True, 'Bile duct stricture', 0.96704006), 'dilated pancreatic ducts': (True, 'Pancreatic duct stricture', 0.9530199), 'calcifications': (True, 'Calcification of muscle', 0.95630056), 'stone fragments': (True, 'Bladder calculus', 0.92041755), 'enlarged pancreas head': (True, 'Pancreatic mass', 0.94890064), 'a forming pseudocyst': (True, 'Right ear pseudocyst', 0.93509805), 'subsequently pancreatitis': (True, 'Acute pancreatitis', 0.96992725), 'all the other abnormal liver function tests': (True, 'Toxic liver disease w cholestasis', 0.88247603), "the patient 's documented Strep septicemia": (True, 'Group B Strep carrier in pregnancy', 0.89074683), 'Streptococcus sanguis': (True, 'Streptococcal pharyngitis', 0.964356), 'this documented septicemia': (True, 'Septic shock', 0.93197143), 'valvular vegetations': (True, 'Aortic valve disease', 0.9303292), 'A complex plaque': (True, 'Stress reaction causing mixed disturbance', 0.90148526), 'this plaque': (True, 'Athetosis', 0.9042806), 'focal mitral valve prolapse': (True, 'Mitral valve prolapse', 0.9814663), 'pathologic mitral regurgitation': (True, 'Mitral valve regurgitation', 0.9347219), 'trace aortic regurgitation': (True, 'Aortic valve regurgitation', 0.97361624), 'his alpha septicemia from his mouth': (True, 'Purulent right mastitis during lactation', 0.91768974), 'a documented alpha Streptococcal septicemia': (True, 'Acute streptococcal tonsillitis', 0.95319), 'Alpha Streptococcus septicemia': (True, 'Streptococcal pharyngitis', 0.9661656), 'resolving right lower lobe pneumonia': (True, 'Right acute recurrent serous otitis media', 0.91119295), 'Type I DM': (True, 'Hx of MI, type 2', 0.92438495), 'hypertensive urgency': (True, 'Hypertensive urgency', 0.9999999), 'his usual nausea': (True, 'Left itchy eye', 0.916708), 'lack of desire': (True, 'Emotional disturbance of childhood', 0.88974667), 'labile blood sugars': (True, 'Low blood pressure reading', 0.9167581), 'critically high': (True, 'Acute MI', 0.9167789), 'sleepiness': (True, 'Sleep talking', 0.97358274), 'localizing signs of infection': (True, 'Localized swelling on head', 0.8968296), 'abscess at the site': (True, 'Abscess of skull bone', 0.9516746), 'GPC in pairs and clusters': (True, 'TB of GI system', 0.9173273), 'DM type': (True, 'Secondary dm', 0.9394595)}

data_tuples = [(key, *value) for key, value in joint_entity_matches.items()]
df = pd.DataFrame(data_tuples, columns=['Entity', 'Match', 'Matched Entity', 'Score'])
df.to_csv('entity_linking_94.csv', index=False)